In [ ]:
# ============================================================
# Stochastic SQP-PINN (JAX) for Nuclear Thermal Coupling (M2PDE Exp2 / MOOSE-style)
# One NN outputs: [phi, Ts, u, v, p, Tf]
#
# FULL VERSION:
#   - block-wise anchor row scaling
#   - region-separated PDE objective/constraints
#   - left BC ramped from IC -> phi.txt
#   - right BC ramped from IC -> 0
#   - top/bottom neutron BC = 0.5
#   - bounded phi with PHI_REF large enough for phi.txt
#   - solid residual scaled consistently with bounded phi
#
# Notes:
#   - SQP tau logic is unchanged
#   - this version fixes the main phi-side compatibility/scaling issues
# ============================================================

import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")


import time, math
from functools import partial
import numpy as np

import jax
import jax.numpy as jnp
from jax import random, vmap, jacrev, hessian

# -----------------------------
# Precision / dtype
# -----------------------------
USE_X64 = True
jax.config.update("jax_enable_x64", USE_X64)
DTYPE = jnp.float64 if USE_X64 else jnp.float32
EPS = DTYPE(1e-12)

# ============================================================
# USER SETTINGS
# ============================================================
Ls = DTYPE(0.0076)
Lf = DTYPE(0.0114)
Ly = DTYPE(0.75)
T_end = DTYPE(5.0)

x_min, x_max = DTYPE(0.0), DTYPE(Ls + Lf)
y_min, y_max = DTYPE(0.0), DTYPE(Ly)
t_min, t_max = DTYPE(0.0), DTYPE(T_end)

V_INLET   = DTYPE(0.4)
T_INLET   = DTYPE(560.0)
P_OUTLET  = DTYPE(0.0)

# neutron BC targets
PHI_TOPBOT_VAL = DTYPE(0.5)
PHI_RIGHT_VAL  = DTYPE(0.0)
T_RAMP_PHI     = DTYPE(0.2)

MU_DAMP_FIXED = DTYPE(1e-4)
X_MARGIN = DTYPE(2e-5)

PHI_TXT_PATH = "./phi.txt"

# -----------------------------
# Scaling targets (block-wise)
# -----------------------------
S_TARGET_PDE   = DTYPE(1.0)
S_TARGET_BC    = DTYPE(1.0)
S_TARGET_IF    = DTYPE(1.0)
S_TARGET_IC    = DTYPE(1.0)
S_ROW_MIN      = DTYPE(1e-6)
S_ROW_MAX      = DTYPE(1.0)   # shrink-only

# ============================================================
# DISCRETIZATION
# ============================================================
NX_OBJ, NY_OBJ, NT_OBJ = 16, 16, 16

NX_CON, NY_CON, NT_CON = 6, 8, 6
K_CON = NX_CON * NY_CON * NT_CON
N_PER_CELL = 1

K_BC = 24
N_BC_PER_BIN = 1

NX_IC, NY_IC = 6, 8
K_IC = NX_IC * NY_IC
N_IC_PER_BIN = 1

NY_IF, NT_IF = 8, 6
K_IF = NY_IF * NT_IF
N_IF_PER_BIN = 1

# block sizes
N_PDE    = 6 * K_CON
N_BC_PHI = 4 * K_BC
N_BC_SOL = 3 * K_BC
N_BC_FLD = 11 * K_BC
N_IF     = 4 * K_IF
N_IC     = 5 * K_IC

M_CON = N_PDE + N_BC_PHI + N_BC_SOL + N_BC_FLD + N_IF + N_IC

# ============================================================
# 1) Load phi.txt + bilinear interp (JIT-safe)
# ============================================================
def load_phi_piecewise_multilinear(path: str):
    with open(path, "r") as f:
        lines = [ln.strip() for ln in f if ln.strip()]

    iy = lines.index("AXIS Y")
    it = lines.index("AXIS T")
    idata = lines.index("DATA")

    y_tokens = []
    k = iy + 1
    while k < len(lines) and lines[k] != "AXIS T":
        for tok in lines[k].split():
            try:
                float(tok)
                y_tokens.append(tok)
            except Exception:
                pass
        k += 1
    y = np.array([float(s) for s in y_tokens], dtype=np.float64)

    t_tokens = []
    k = it + 1
    while k < len(lines) and lines[k] != "DATA":
        for tok in lines[k].split():
            try:
                float(tok)
                t_tokens.append(tok)
            except Exception:
                pass
        k += 1
    t = np.array([float(s) for s in t_tokens], dtype=np.float64)

    Ny = len(y)
    Nt = len(t)

    data_tokens = []
    for ln in lines[idata + 1:]:
        for tok in ln.split():
            data_tokens.append(float(tok))
    data = np.array(data_tokens, dtype=np.float64)

    if data.size != Ny * Nt:
        raise ValueError(
            f"phi.txt DATA size mismatch: got {data.size}, expected {Ny*Nt} (Ny={Ny}, Nt={Nt})"
        )

    Z_t_y = data.reshape(Nt, Ny)
    Z_y_t = Z_t_y.T
    return y, t, Z_y_t


y_np, t_np, Z_np = load_phi_piecewise_multilinear(PHI_TXT_PATH)
PHI_Y = jnp.array(y_np, dtype=DTYPE)
PHI_T = jnp.array(t_np, dtype=DTYPE)
PHI_Z = jnp.array(Z_np, dtype=DTYPE)

print(
    "Loaded phi.txt:",
    y_np.shape,
    t_np.shape,
    Z_np.shape,
    "range=",
    float(Z_np.min()),
    float(Z_np.max()),
)

@jax.jit
def phi_bc_yt(y, t, y_grid, t_grid, Z):
    y = jnp.asarray(y, dtype=DTYPE)
    t = jnp.asarray(t, dtype=DTYPE)

    Ny = y_grid.shape[0]
    Nt = t_grid.shape[0]

    iy = jnp.clip(jnp.searchsorted(y_grid, y, side="right") - 1, 0, Ny - 2)
    it = jnp.clip(jnp.searchsorted(t_grid, t, side="right") - 1, 0, Nt - 2)

    y0 = y_grid[iy]
    y1 = y_grid[iy + 1]
    t0 = t_grid[it]
    t1 = t_grid[it + 1]

    wy = (y - y0) / (y1 - y0 + EPS)
    wt = (t - t0) / (t1 - t0 + EPS)

    z00 = Z[iy,     it]
    z10 = Z[iy + 1, it]
    z01 = Z[iy,     it + 1]
    z11 = Z[iy + 1, it + 1]

    z0 = z00 + wy * (z10 - z00)
    z1 = z01 + wy * (z11 - z01)
    return z0 + wt * (z1 - z0)

# ============================================================
# 2) Material laws
# ============================================================
rho_f = DTYPE(11096.0)
rho_s = DTYPE(16020.0)

def safe_T(T):
    return jnp.clip(T, DTYPE(300.0), DTYPE(1200.0))

def mu_f_fun(T):
    T = safe_T(T)
    return DTYPE(4.94e-4) * jnp.exp(DTYPE(754.1) / (T + EPS))

def k_f_fun(T):
    T = safe_T(T)
    return DTYPE(3.61) + DTYPE(1.517e-2) * T - DTYPE(1.741e-6) * T * T

def cp_f_fun(T):
    T = safe_T(T)
    return DTYPE(159.0) - DTYPE(2.72e-2) * T + DTYPE(7.12e-6) * T * T

def cp_s_fun(T):
    T = safe_T(T)
    return (DTYPE(1.359) + DTYPE(0.05812) * T + DTYPE(1.086e6) / (T * T + EPS)) * DTYPE(5.0)

def k_s_fun(T):
    T = safe_T(T)
    c0 = DTYPE(17.5) * (DTYPE(1.0) - DTYPE(0.223)) / (DTYPE(1.0) + DTYPE(0.161))
    c1 = DTYPE(1.54e-2) * (DTYPE(1.0) + DTYPE(0.0061)) / (DTYPE(1.0) + DTYPE(0.161))
    c2 = DTYPE(9.38e-6)
    return c0 + c1 * T + c2 * T * T

POWER_COEF = DTYPE(5e7)

# Neutron
v_neu   = DTYPE(2.416)
D_fuel  = DTYPE(0.008249)
D_fluid = DTYPE(0.01)

def sigma_af_fuel(T):
    T = safe_T(T)
    return (
        DTYPE(2.416) * DTYPE(583.5) * DTYPE(1.305) * DTYPE(1.602) * DTYPE(0.1)
        - (DTYPE(13.47) * (T - DTYPE(560.0)) / (DTYPE(900.0) - DTYPE(560.0)) + DTYPE(7.53))
          * DTYPE(2.1479) * DTYPE(1.602)
        + DTYPE(0.185) * DTYPE(6.6072) * DTYPE(0.1)
        - DTYPE(680.9) * DTYPE(1.305) * DTYPE(1.602) * DTYPE(0.1)
    )

def sigma_af_fluid(T):
    T = safe_T(T)
    return -(DTYPE(20.0) + DTYPE(20.0) * (T - DTYPE(560.0)) / (DTYPE(800.0) - DTYPE(560.0)))

# ---- variable-coefficient diffusion helpers ----
def dk_s_dT(T):
    T = safe_T(T)
    c1 = DTYPE(1.54e-2) * (DTYPE(1.0) + DTYPE(0.0061)) / (DTYPE(1.0) + DTYPE(0.161))
    c2 = DTYPE(9.38e-6)
    return c1 + DTYPE(2.0) * c2 * T

def dk_f_dT(T):
    T = safe_T(T)
    a = DTYPE(1.517e-2)
    b = DTYPE(1.741e-6)
    return a - DTYPE(2.0) * b * T

def dmu_f_dT(T):
    T = safe_T(T)
    mu = mu_f_fun(T)
    return mu * (-DTYPE(754.1)) / (T * T + EPS)

def div_k_grad_scalar(k, dk_dT_fun, T, Tx, Ty, Txx, Tyy):
    # div(k(T) grad T) = k*(Txx+Tyy) + dk/dT*(Tx^2+Ty^2)
    return k * (Txx + Tyy) + dk_dT_fun(T) * (Tx * Tx + Ty * Ty)

def div_mu_grad_component(mu, dmu_dT_fun, Tf, Tfx, Tfy, ux, uy, uxx, uyy):
    # div(mu(Tf) grad u) = mu*(uxx+uyy) + dmu/dT*(Tfx*ux + Tfy*uy)
    return mu * (uxx + uyy) + dmu_dT_fun(Tf) * (Tfx * ux + Tfy * uy)

# ============================================================
# Physics-based reference scales
# ============================================================
Lx       = (x_max - x_min) + EPS
Ly_len   = (y_max - y_min) + EPS
Lx_fluid = (x_max - Ls) + EPS
Lx_solid = (Ls - x_min) + EPS

T_SCALE = DTYPE(100.0)
U_SCALE = V_INLET + EPS
P_SCALE = rho_f * (V_INLET ** 2) + EPS

t_ref = Ly_len / U_SCALE

# Must be large enough to cover phi.txt max (~6.2)
PHI_REF = DTYPE(7.0)

D_MAX   = jnp.maximum(D_fuel, D_fluid)
SIG_REF = jnp.maximum(jnp.abs(sigma_af_fuel(T_INLET)), jnp.abs(sigma_af_fluid(T_INLET)))
Lmin    = jnp.minimum(Lx, Ly_len)
R_diff  = D_MAX * PHI_REF / (Lmin ** 2 + EPS)
R_reac  = SIG_REF * PHI_REF
R_time  = PHI_REF / (v_neu * (t_ref + EPS))
PHI_SCALE = R_diff + R_reac + R_time + EPS

GRAD_UY_SCALE  = U_SCALE / Ly_len
GRAD_TY_SCALE  = T_SCALE / Ly_len
GRAD_UX_SCALE  = U_SCALE / Lx_fluid
GRAD_TX_SCALE  = T_SCALE / Lx_fluid
GRAD_PX_SCALE  = P_SCALE / Lx_fluid
GRAD_TSX_SCALE = T_SCALE / Lx_solid

K_REF      = k_s_fun(T_INLET)
DT_REF     = DTYPE(100.0)
FLUX_SCALE = K_REF * DT_REF / Lx_fluid + EPS

CONT_SCALE = U_SCALE / (jnp.minimum(Lx_fluid, Ly_len) + EPS)

# ============================================================
# Targets (phi IC + ramped BC)
# ============================================================
@jax.jit
# ============================================================
# Targets (phi IC + hard BC, no ramp)
# ============================================================
@jax.jit
def phi_ic(y):
    return DTYPE(0.5) + DTYPE(1.5) * jnp.cos(DTYPE(jnp.pi) * (y - DTYPE(0.375)) / DTYPE(0.75))

def phi_left_target(y, t):
    # hard left BC from phi.txt for all t
    t_clip = jnp.clip(t, PHI_T[0], PHI_T[-1])
    return phi_bc_yt(y, t_clip, PHI_Y, PHI_T, PHI_Z)

def phi_right_target(y, t):
    # hard right BC = 0 for all t
    return PHI_RIGHT_VAL * jnp.ones_like(y)

def phi_y_target(y_fixed, t):
    # hard top/bottom BC = 0.5 for all t
    return PHI_TOPBOT_VAL * jnp.ones_like(t)

def inlet_v_profile(x):
    return V_INLET * jnp.ones_like(x)

# ============================================================
# 4) MLP
# ============================================================
def init_mlp_params(key, layer_sizes):
    params = []
    keys = random.split(key, len(layer_sizes) - 1)
    for k, (m, n) in zip(keys, zip(layer_sizes[:-1], layer_sizes[1:])):
        W = random.normal(k, (m, n), dtype=DTYPE) * jnp.sqrt(DTYPE(2.0) / DTYPE(m))
        b = jnp.zeros((n,), dtype=DTYPE)
        params.append({"W": W, "b": b})
    return params

def mlp_apply(params, X):
    """
    Network outputs are dimensionless -> map to physical here.

    Outputs:
      phi = PHI_REF * tanh(phi_hat)
      Ts  = T_INLET + TAMP * tanh(Ts_hat)
      u,v = U_SCALE * tanh(u_hat, v_hat)
      p   = P_SCALE * tanh(p_hat)
      Tf  = T_INLET + TAMP * tanh(Tf_hat)
    """
    h = X
    for i, layer in enumerate(params):
        W, b = layer["W"], layer["b"]
        h = h @ W + b
        if i < len(params) - 1:
            h = jnp.tanh(h)

    phi_hat = h[:, 0:1]
    Ts_hat  = h[:, 1:2]
    u_hat   = h[:, 2:3]
    v_hat   = h[:, 3:4]
    p_hat   = h[:, 4:5]
    Tf_hat  = h[:, 5:6]

    phi = PHI_REF * jnp.tanh(phi_hat)

    TAMP = DTYPE(20.0)
    Ts = T_INLET + TAMP * jnp.tanh(Ts_hat)
    Tf = T_INLET + TAMP * jnp.tanh(Tf_hat)

    u = U_SCALE * jnp.tanh(u_hat)
    v = U_SCALE * jnp.tanh(v_hat)
    p = P_SCALE * jnp.tanh(p_hat)

    return jnp.concatenate([phi, Ts, u, v, p, Tf], axis=1)

def flatten_params(params):
    flat_parts = []
    shapes = []
    for layer in params:
        W, b = layer["W"], layer["b"]
        flat_parts.append(W.reshape(-1))
        flat_parts.append(b.reshape(-1))
        shapes.append((W.shape, b.shape))
    theta = jnp.concatenate(flat_parts).astype(DTYPE)
    return theta, tuple(shapes)

def unflatten_params(theta, shapes):
    params = []
    idx = 0
    for W_shape, b_shape in shapes:
        W_size = math.prod(W_shape)
        b_size = math.prod(b_shape)
        W = theta[idx: idx + W_size].reshape(W_shape)
        idx += W_size
        b = theta[idx: idx + b_size].reshape(b_shape)
        idx += b_size
        params.append({"W": W, "b": b})
    return params

# ============================================================
# segment sum
# ============================================================
def segment_sum(values, segment_ids, num_segments):
    out = jnp.zeros((num_segments,), dtype=values.dtype)
    return out.at[segment_ids].add(values)

# ============================================================
# Sampling helpers
# ============================================================
def sample_stratified_3d(key, x0, x1, y0, y1, t0, t1, NX, NY, NT):
    dx = (DTYPE(x1) - DTYPE(x0)) / DTYPE(NX)
    dy = (DTYPE(y1) - DTYPE(y0)) / DTYPE(NY)
    dt = (DTYPE(t1) - DTYPE(t0)) / DTYPE(NT)

    it, iy, ix = jnp.meshgrid(jnp.arange(NT), jnp.arange(NY), jnp.arange(NX), indexing="ij")
    it = it.reshape(-1).astype(jnp.int32)
    iy = iy.reshape(-1).astype(jnp.int32)
    ix = ix.reshape(-1).astype(jnp.int32)
    ids = (it * (NY * NX) + iy * NX + ix).astype(jnp.int32)

    xb = DTYPE(x0) + DTYPE(ix) * dx
    yb = DTYPE(y0) + DTYPE(iy) * dy
    tb = DTYPE(t0) + DTYPE(it) * dt

    K = NX * NY * NT
    u = random.uniform(key, (K, 3), minval=0.0, maxval=1.0, dtype=DTYPE)
    xs = xb + u[:, 0] * dx
    ys = yb + u[:, 1] * dy
    ts = tb + u[:, 2] * dt
    X = jnp.stack([xs, ys, ts], axis=1)
    return X, ids

def sample_bc_time_binned(key, K, *, x_fixed=None, y_fixed=None, t0=t_min, t1=t_max,
                          x_lo=None, x_hi=None, y_lo=None, y_hi=None):
    dt = (DTYPE(t1) - DTYPE(t0)) / DTYPE(K)
    j = jnp.arange(K, dtype=jnp.int32)
    tb = DTYPE(t0) + DTYPE(j) * dt

    key, kt, kr = random.split(key, 3)
    u_t = random.uniform(kt, (K, N_BC_PER_BIN), minval=0.0, maxval=1.0, dtype=DTYPE)
    ts = (tb[:, None] + u_t * dt).reshape(-1, 1)
    ids = jnp.repeat(j, N_BC_PER_BIN)

    if x_fixed is not None:
        lo = DTYPE(y_min if y_lo is None else y_lo)
        hi = DTYPE(y_max if y_hi is None else y_hi)
        y = random.uniform(kr, (ts.shape[0], 1), minval=lo, maxval=hi, dtype=DTYPE)
        x = DTYPE(x_fixed) * jnp.ones_like(y)
        return jnp.concatenate([x, y, ts], axis=1), ids

    if y_fixed is not None:
        lo = DTYPE(x_min if x_lo is None else x_lo)
        hi = DTYPE(x_max if x_hi is None else x_hi)
        x = random.uniform(kr, (ts.shape[0], 1), minval=lo, maxval=hi, dtype=DTYPE)
        y = DTYPE(y_fixed) * jnp.ones_like(x)
        return jnp.concatenate([x, y, ts], axis=1), ids

    raise ValueError("Provide x_fixed or y_fixed")

def sample_ic_xy(key):
    y0 = DTYPE(0.02)
    y1 = y_max - DTYPE(0.02)

    dx = (x_max - x_min) / DTYPE(NX_IC)
    dy = (y1 - y0) / DTYPE(NY_IC)

    iy, ix = jnp.meshgrid(jnp.arange(NY_IC), jnp.arange(NX_IC), indexing="ij")
    iy = iy.reshape(-1).astype(jnp.int32)
    ix = ix.reshape(-1).astype(jnp.int32)
    ids0 = (iy * NX_IC + ix).astype(jnp.int32)

    xb = x_min + DTYPE(ix) * dx
    yb = y0 + DTYPE(iy) * dy

    u = random.uniform(key, (K_IC, N_IC_PER_BIN, 2), minval=0.0, maxval=1.0, dtype=DTYPE)
    xs = (xb[:, None] + u[:, :, 0] * dx).reshape(-1, 1)
    ys = (yb[:, None] + u[:, :, 1] * dy).reshape(-1, 1)
    ts = t_min * jnp.ones_like(xs)

    X = jnp.concatenate([xs, ys, ts], axis=1)
    ids = jnp.repeat(ids0, N_IC_PER_BIN)
    return X, ids

def sample_interface_yt(key):
    dy = (y_max - y_min) / DTYPE(NY_IF)
    dt = (t_max - t_min) / DTYPE(NT_IF)

    it, iy = jnp.meshgrid(jnp.arange(NT_IF), jnp.arange(NY_IF), indexing="ij")
    it = it.reshape(-1).astype(jnp.int32)
    iy = iy.reshape(-1).astype(jnp.int32)
    ids0 = (it * NY_IF + iy).astype(jnp.int32)

    yb = y_min + DTYPE(iy) * dy
    tb = t_min + DTYPE(it) * dt

    u = random.uniform(key, (NY_IF * NT_IF, N_IF_PER_BIN, 2), minval=0.0, maxval=1.0, dtype=DTYPE)
    ys = (yb[:, None] + u[:, :, 0] * dy).reshape(-1, 1)
    ts = (tb[:, None] + u[:, :, 1] * dt).reshape(-1, 1)
    xs = Ls * jnp.ones_like(ys)

    X = jnp.concatenate([xs, ys, ts], axis=1)
    ids = jnp.repeat(ids0, N_IF_PER_BIN)
    return X, ids

# ============================================================
# Jitter + projection
# ============================================================
def jitter_xyt(key, X, sx, sy, st, clip_k=2.0):
    if (sx <= 0) and (sy <= 0) and (st <= 0):
        return X
    key, kx, ky, kt = random.split(key, 4)
    dx = DTYPE(sx) * random.normal(kx, (X.shape[0],), dtype=DTYPE)
    dy = DTYPE(sy) * random.normal(ky, (X.shape[0],), dtype=DTYPE)
    dt = DTYPE(st) * random.normal(kt, (X.shape[0],), dtype=DTYPE)
    k = DTYPE(clip_k)
    dx = jnp.clip(dx, -k * DTYPE(sx), +k * DTYPE(sx))
    dy = jnp.clip(dy, -k * DTYPE(sy), +k * DTYPE(sy))
    dt = jnp.clip(dt, -k * DTYPE(st), +k * DTYPE(st))
    x = jnp.clip(X[:, 0] + dx, x_min, x_max)
    y = jnp.clip(X[:, 1] + dy, y_min, y_max)
    t = jnp.clip(X[:, 2] + dt, t_min, t_max)
    return jnp.stack([x, y, t], axis=1)

def project_x_fixed(X, x_fixed):
    return jnp.stack(
        [
            DTYPE(x_fixed) * jnp.ones((X.shape[0],), dtype=DTYPE),
            X[:, 1],
            X[:, 2],
        ],
        axis=1,
    )

def project_y_fixed(X, y_fixed, *, x_lo=None, x_hi=None):
    x = X[:, 0]
    if (x_lo is not None) or (x_hi is not None):
        lo = DTYPE(x_lo if x_lo is not None else x_min)
        hi = DTYPE(x_hi if x_hi is not None else x_max)
        x = jnp.clip(x, lo, hi)
    y = DTYPE(y_fixed) * jnp.ones((X.shape[0],), dtype=DTYPE)
    return jnp.stack([x, y, X[:, 2]], axis=1)

def project_t0(X):
    return jnp.stack(
        [X[:, 0], X[:, 1], t_min * jnp.ones((X.shape[0],), dtype=DTYPE)],
        axis=1,
    )

def project_interface(X):
    return project_x_fixed(X, Ls)

# ============================================================
# PDE residuals
# ============================================================
def forward6(params, xyt):
    return mlp_apply(params, xyt[None, :])[0, :]

def eval_fields_and_derivs(params, X):
    out = vmap(lambda z: forward6(params, z))(X)
    J   = vmap(jacrev(lambda z: forward6(params, z)))(X)
    H   = vmap(hessian(lambda z: forward6(params, z)))(X)
    return out, J, H

def residuals_all(params, X):
    out, J, H = eval_fields_and_derivs(params, X)

    phi = out[:, 0]
    Ts  = out[:, 1]
    u   = out[:, 2]
    v   = out[:, 3]
    p   = out[:, 4]
    Tf  = out[:, 5]

    x = X[:, 0]
    m_s = (x <= Ls).astype(DTYPE)
    m_f = (x >  Ls).astype(DTYPE)

    # neutron
    phi_t  = J[:, 0, 2]
    phi_xx = H[:, 0, 0, 0]
    phi_yy = H[:, 0, 1, 1]
    lap_phi = phi_xx + phi_yy

    # solid
    Ts_x  = J[:, 1, 0]
    Ts_y  = J[:, 1, 1]
    Ts_t  = J[:, 1, 2]
    Ts_xx = H[:, 1, 0, 0]
    Ts_yy = H[:, 1, 1, 1]

    # fluid
    u_x, u_y, u_t = J[:, 2, 0], J[:, 2, 1], J[:, 2, 2]
    v_x, v_y, v_t = J[:, 3, 0], J[:, 3, 1], J[:, 3, 2]
    p_x, p_y      = J[:, 4, 0], J[:, 4, 1]
    Tf_x, Tf_y, Tf_t = J[:, 5, 0], J[:, 5, 1], J[:, 5, 2]

    u_xx, u_yy   = H[:, 2, 0, 0], H[:, 2, 1, 1]
    v_xx, v_yy   = H[:, 3, 0, 0], H[:, 3, 1, 1]
    Tf_xx, Tf_yy = H[:, 5, 0, 0], H[:, 5, 1, 1]

    is_solid = (x <= Ls)
    T_for_sigma = jnp.where(is_solid, Ts, Tf)
    D_here      = jnp.where(is_solid, D_fuel, D_fluid)
    sig_fuel    = sigma_af_fuel(T_for_sigma)
    sig_fluid   = sigma_af_fluid(T_for_sigma)
    sig_here    = jnp.where(is_solid, sig_fuel, sig_fluid)

    r_phi_raw = (DTYPE(1.0) / v_neu) * phi_t - D_here * lap_phi - sig_here * phi
    r_phi = r_phi_raw / PHI_SCALE

    div_k_grad_Ts = div_k_grad_scalar(k_s_fun(Ts), dk_s_dT, Ts, Ts_x, Ts_y, Ts_xx, Ts_yy)
    r_Ts_raw = rho_s * cp_s_fun(Ts) * Ts_t - div_k_grad_Ts - POWER_COEF * phi
    r_Ts = r_Ts_raw / (POWER_COEF * PHI_REF + EPS)

    r_cont = (u_x + v_y) / (CONT_SCALE + EPS)

    mu_here = mu_f_fun(Tf)
    visc_u = div_mu_grad_component(mu_here, dmu_f_dT, Tf, Tf_x, Tf_y, u_x, u_y, u_xx, u_yy)
    visc_v = div_mu_grad_component(mu_here, dmu_f_dT, Tf, Tf_x, Tf_y, v_x, v_y, v_xx, v_yy)

    r_u_raw = rho_f * (u_t + u * u_x + v * u_y) + p_x - visc_u
    r_v_raw = rho_f * (v_t + u * v_x + v * v_y) + p_y - visc_v

    cp_here = cp_f_fun(Tf)
    div_k_grad_Tf = div_k_grad_scalar(k_f_fun(Tf), dk_f_dT, Tf, Tf_x, Tf_y, Tf_xx, Tf_yy)
    r_Tf_raw = rho_f * cp_here * (Tf_t + u * Tf_x + v * Tf_y) - div_k_grad_Tf

    r_u  = r_u_raw / rho_f
    r_v  = r_v_raw / rho_f
    r_Tf = r_Tf_raw / (rho_f * cp_here + EPS)

    return r_phi, r_Ts, r_cont, r_u, r_v, r_Tf, m_s, m_f

# ============================================================
# Objective: PDE residual MSE on objective points
# ============================================================
@partial(jax.jit, static_argnames=("shapes",))
def F_and_g_pde_obj(theta, shapes, X_obj):
    def obj_theta(th):
        params = unflatten_params(th, shapes)
        r_phi, r_Ts, r_cont, r_u, r_v, r_Tf, m_s, m_f = residuals_all(params, X_obj)
        term = (r_phi ** 2) + m_s * (r_Ts ** 2) + m_f * (r_cont ** 2 + r_u ** 2 + r_v ** 2 + r_Tf ** 2)
        return jnp.mean(term)
    val, g = jax.value_and_grad(obj_theta)(theta)
    return val, g

# ============================================================
# Constraint vector (UNSCALED) + Jacobian
# ============================================================
@partial(jax.jit, static_argnames=("shapes",))
def constraint_vector(theta, shapes,
                      X_con, ids_con,
                      X_phi_left, ids_phi_left,
                      X_phi_right, ids_phi_right,
                      X_phi_y0, ids_phi_y0,
                      X_phi_y1, ids_phi_y1,
                      X_Ts_x0, ids_Ts_x0,
                      X_Ts_y0, ids_Ts_y0,
                      X_Ts_y1, ids_Ts_y1,
                      X_f_inlet, ids_f_inlet,
                      X_f_outlet, ids_f_outlet,
                      X_f_right, ids_f_right,
                      X_if, ids_if,
                      X_ic, ids_ic):
    params = unflatten_params(theta, shapes)

    r_phi, r_Ts, r_cont, r_u, r_v, r_Tf, m_s, m_f = residuals_all(params, X_con)

    # PDE constraints (binned)
    c_phi  = segment_sum(r_phi,        ids_con, K_CON) / DTYPE(N_PER_CELL)
    c_Ts   = segment_sum(m_s * r_Ts,   ids_con, K_CON) / DTYPE(N_PER_CELL)
    c_cont = segment_sum(m_f * r_cont, ids_con, K_CON) / DTYPE(N_PER_CELL)
    c_ru   = segment_sum(m_f * r_u,    ids_con, K_CON) / DTYPE(N_PER_CELL)
    c_rv   = segment_sum(m_f * r_v,    ids_con, K_CON) / DTYPE(N_PER_CELL)
    c_rTf  = segment_sum(m_f * r_Tf,   ids_con, K_CON) / DTYPE(N_PER_CELL)

    c_pde = jnp.concatenate([c_phi, c_Ts, c_cont, c_ru, c_rv, c_rTf], axis=0)

    # -------------------------
    # Neutron BCs (4*K_BC)
    # -------------------------
    phiL = mlp_apply(params, X_phi_left)[:, 0]
    yL, tL = X_phi_left[:, 1], X_phi_left[:, 2]
    c_phi_left = segment_sum((phiL - phi_left_target(yL, tL)) / PHI_REF, ids_phi_left, K_BC) / DTYPE(N_BC_PER_BIN)

    phiR = mlp_apply(params, X_phi_right)[:, 0]
    yR, tR = X_phi_right[:, 1], X_phi_right[:, 2]
    c_phi_right = segment_sum((phiR - phi_right_target(yR, tR)) / PHI_REF, ids_phi_right, K_BC) / DTYPE(N_BC_PER_BIN)

    phiB = mlp_apply(params, X_phi_y0)[:, 0]
    tB = X_phi_y0[:, 2]
    c_phi_y0 = segment_sum((phiB - phi_y_target(y_min, tB)) / PHI_REF, ids_phi_y0, K_BC) / DTYPE(N_BC_PER_BIN)

    phiT = mlp_apply(params, X_phi_y1)[:, 0]
    tT = X_phi_y1[:, 2]
    c_phi_y1 = segment_sum((phiT - phi_y_target(y_max, tT)) / PHI_REF, ids_phi_y1, K_BC) / DTYPE(N_BC_PER_BIN)

    c_bc_phi = jnp.concatenate([c_phi_left, c_phi_right, c_phi_y0, c_phi_y1], axis=0)

    # -------------------------
    # Solid BCs (3*K_BC)
    # -------------------------
    def Ts_fun(z):
        return forward6(params, z)[1]

    Tsx_x0 = vmap(lambda z: jacrev(Ts_fun)(z)[0])(X_Ts_x0)
    c_Ts_x0 = segment_sum(Tsx_x0 / GRAD_TSX_SCALE, ids_Ts_x0, K_BC) / DTYPE(N_BC_PER_BIN)

    Tsy_y0 = vmap(lambda z: jacrev(Ts_fun)(z)[1])(X_Ts_y0)
    Tsy_y1 = vmap(lambda z: jacrev(Ts_fun)(z)[1])(X_Ts_y1)
    c_Ts_y0 = segment_sum(Tsy_y0 / GRAD_TY_SCALE, ids_Ts_y0, K_BC) / DTYPE(N_BC_PER_BIN)
    c_Ts_y1 = segment_sum(Tsy_y1 / GRAD_TY_SCALE, ids_Ts_y1, K_BC) / DTYPE(N_BC_PER_BIN)

    c_solid_bc = jnp.concatenate([c_Ts_x0, c_Ts_y0, c_Ts_y1], axis=0)

    # -------------------------
    # Fluid BCs (11*K_BC)
    # -------------------------
    def u_fun(z):
        return forward6(params, z)[2]

    def v_fun(z):
        return forward6(params, z)[3]

    def p_fun(z):
        return forward6(params, z)[4]

    def Tf_fun(z):
        return forward6(params, z)[5]

    # inlet y=0
    out_in = mlp_apply(params, X_f_inlet)
    u_in, v_in, Tf_in = out_in[:, 2], out_in[:, 3], out_in[:, 5]
    x_in = X_f_inlet[:, 0]
    v_tar = inlet_v_profile(x_in)

    c_in_u  = segment_sum((u_in - DTYPE(0.0)) / U_SCALE, ids_f_inlet, K_BC) / DTYPE(N_BC_PER_BIN)
    c_in_v  = segment_sum((v_in - v_tar) / U_SCALE,      ids_f_inlet, K_BC) / DTYPE(N_BC_PER_BIN)
    c_in_Tf = segment_sum((Tf_in - T_INLET) / T_SCALE,   ids_f_inlet, K_BC) / DTYPE(N_BC_PER_BIN)
    c_inlet = jnp.concatenate([c_in_u, c_in_v, c_in_Tf], axis=0)

    # outlet y=Ly
    p_out = mlp_apply(params, X_f_outlet)[:, 4]
    c_out_p = segment_sum((p_out - P_OUTLET) / P_SCALE, ids_f_outlet, K_BC) / DTYPE(N_BC_PER_BIN)

    uy_out  = vmap(lambda z: jacrev(u_fun)(z)[1])(X_f_outlet)
    vy_out  = vmap(lambda z: jacrev(v_fun)(z)[1])(X_f_outlet)
    Tfy_out = vmap(lambda z: jacrev(Tf_fun)(z)[1])(X_f_outlet)

    c_out_uy  = segment_sum(uy_out  / GRAD_UY_SCALE, ids_f_outlet, K_BC) / DTYPE(N_BC_PER_BIN)
    c_out_vy  = segment_sum(vy_out  / GRAD_UY_SCALE, ids_f_outlet, K_BC) / DTYPE(N_BC_PER_BIN)
    c_out_Tfy = segment_sum(Tfy_out / GRAD_TY_SCALE, ids_f_outlet, K_BC) / DTYPE(N_BC_PER_BIN)
    c_outlet = jnp.concatenate([c_out_p, c_out_uy, c_out_vy, c_out_Tfy], axis=0)

    # right symmetry x=x_max: u=0, v_x=0, p_x=0, Tf_x=0
    u_r = mlp_apply(params, X_f_right)[:, 2]
    vx_r  = vmap(lambda z: jacrev(v_fun)(z)[0])(X_f_right)
    px_r  = vmap(lambda z: jacrev(p_fun)(z)[0])(X_f_right)
    Tfx_r = vmap(lambda z: jacrev(Tf_fun)(z)[0])(X_f_right)

    c_r_u  = segment_sum(u_r / U_SCALE,       ids_f_right, K_BC) / DTYPE(N_BC_PER_BIN)
    c_r_vx = segment_sum(vx_r / GRAD_UX_SCALE, ids_f_right, K_BC) / DTYPE(N_BC_PER_BIN)
    c_r_px = segment_sum(px_r / GRAD_PX_SCALE, ids_f_right, K_BC) / DTYPE(N_BC_PER_BIN)
    c_r_Tx = segment_sum(Tfx_r / GRAD_TX_SCALE, ids_f_right, K_BC) / DTYPE(N_BC_PER_BIN)
    c_sym = jnp.concatenate([c_r_u, c_r_vx, c_r_px, c_r_Tx], axis=0)

    c_bc_f = jnp.concatenate([c_inlet, c_outlet, c_sym], axis=0)

    # -------------------------
    # Interface (4*K_IF)
    # -------------------------
    out_if = mlp_apply(params, X_if)
    Ts_if, Tf_if = out_if[:, 1], out_if[:, 5]
    u_if, v_if   = out_if[:, 2], out_if[:, 3]

    c_if_T = segment_sum((Ts_if - Tf_if) / T_SCALE, ids_if, K_IF) / DTYPE(N_IF_PER_BIN)

    Tsx_if = vmap(lambda z: jacrev(Ts_fun)(z)[0])(X_if)
    Tfx_if = vmap(lambda z: jacrev(Tf_fun)(z)[0])(X_if)

    q_s = -k_s_fun(Ts_if) * Tsx_if
    q_f = -k_f_fun(Tf_if) * Tfx_if
    c_if_q = segment_sum((q_s - q_f) / FLUX_SCALE, ids_if, K_IF) / DTYPE(N_IF_PER_BIN)

    c_if_u = segment_sum(u_if / U_SCALE, ids_if, K_IF) / DTYPE(N_IF_PER_BIN)
    c_if_v = segment_sum(v_if / U_SCALE, ids_if, K_IF) / DTYPE(N_IF_PER_BIN)
    c_if = jnp.concatenate([c_if_T, c_if_q, c_if_u, c_if_v], axis=0)

    # -------------------------
    # IC (5*K_IC)
    # -------------------------
    out0 = mlp_apply(params, X_ic)
    phi0 = out0[:, 0]
    Ts0  = out0[:, 1]
    u0   = out0[:, 2]
    v0   = out0[:, 3]
    Tf0  = out0[:, 5]

    x0 = X_ic[:, 0]
    m_s0 = (x0 <= Ls).astype(DTYPE)
    m_f0 = (x0 >  Ls).astype(DTYPE)

    y0 = X_ic[:, 1]
    c_ic_phi = segment_sum((phi0 - phi_ic(y0)) / PHI_REF, ids_ic, K_IC) / DTYPE(N_IC_PER_BIN)
    c_ic_Ts  = segment_sum(m_s0 * ((Ts0 - T_INLET) / T_SCALE), ids_ic, K_IC) / DTYPE(N_IC_PER_BIN)
    c_ic_u   = segment_sum(m_f0 * ((u0  - DTYPE(0.0)) / U_SCALE), ids_ic, K_IC) / DTYPE(N_IC_PER_BIN)
    c_ic_v   = segment_sum(m_f0 * ((v0  - DTYPE(0.0)) / U_SCALE), ids_ic, K_IC) / DTYPE(N_IC_PER_BIN)
    c_ic_Tf  = segment_sum(m_f0 * ((Tf0 - T_INLET) / T_SCALE), ids_ic, K_IC) / DTYPE(N_IC_PER_BIN)

    c_ic = jnp.concatenate([c_ic_phi, c_ic_Ts, c_ic_u, c_ic_v, c_ic_Tf], axis=0)

    return jnp.concatenate([c_pde, c_bc_phi, c_solid_bc, c_bc_f, c_if, c_ic], axis=0)

@partial(jax.jit, static_argnames=("shapes",))
def C_and_J(theta, shapes, *args):
    def c_fun(th):
        return constraint_vector(th, shapes, *args)

    def c_fun_aux(th):
        c = c_fun(th)
        return c, c

    J, c = jax.jacrev(c_fun_aux, has_aux=True)(theta)
    return c, J

@partial(jax.jit, static_argnames=("shapes",))
def C_only(theta, shapes, *args):
    return constraint_vector(theta, shapes, *args)

# ============================================================
# KKT solve
# ============================================================
@jax.jit
def kkt_solve_once(H, Jac, Grad, Cons, mu_damp):
    n = H.shape[0]
    m = Cons.shape[0]
    I_m = jnp.eye(m, dtype=H.dtype)
    top = jnp.concatenate([H, Jac.T], axis=1)
    bottom = jnp.concatenate([Jac, -mu_damp * I_m], axis=1)
    KKT = jnp.concatenate([top, bottom], axis=0)
    rhs = -jnp.concatenate([Grad, Cons])
    sol = jnp.linalg.solve(KKT, rhs)
    d = sol[:n]
    y = sol[n:]
    return d, y

# ============================================================
# Zhou-style helpers (unchanged SQP tau logic)
# ============================================================
def cal_tau_mu(H, d, sigma, tau_pre, eps_tau, g, c, mu_eff, y):
    denom = float(g @ d + 0.5 * (d @ (H @ d)))
    c1 = float(jnp.linalg.norm(c, 1))
    mu_y1 = float(jnp.linalg.norm(mu_eff * y, 1))
    delta_c = c1 - mu_y1
    if (denom <= 1e-12) or (delta_c <= 0.0):
        tau_trial = float("inf")
    else:
        tau_trial = (1.0 - sigma) * delta_c / denom
    if tau_pre <= tau_trial:
        return tau_pre
    return min(tau_trial, (1.0 - eps_tau) * tau_pre)

def cal_ksi_mu(d, tau, ksi_old, eps_ksi, g, c, mu_eff, y):
    d2 = float(jnp.linalg.norm(d) ** 2)
    if d2 <= 1e-18 or tau <= 1e-18:
        return ksi_old
    gd = float(g @ d)
    c1 = float(jnp.linalg.norm(c, 1))
    mu_y1 = float(jnp.linalg.norm(mu_eff * y, 1))
    delta_c = c1 - mu_y1
    Dl = -tau * gd + delta_c
    ksi_trial = Dl / (tau * d2)
    ksi_trial = max(0.0, ksi_trial)
    if ksi_old <= ksi_trial:
        return ksi_old
    return min(ksi_trial, (1.0 - eps_ksi) * ksi_old)

def phi_mu(alpha, eta, beta, tau, g, d, c, mu_eff, y, L, Gamma):
    gd = float(g @ d)
    d2 = float(d @ d)
    c1 = float(jnp.linalg.norm(c, 1))
    mu_y1 = float(jnp.linalg.norm(mu_eff * y, 1))
    delta_c = c1 - mu_y1
    Dl = -tau * gd + delta_c
    term1 = (eta - 1.0) * alpha * beta * Dl
    term2 = (abs(1.0 - alpha) - 1.0 + alpha) * c1
    term3 = 0.5 * (tau * L + Gamma) * (alpha ** 2) * d2
    return term1 + term2 + term3

def cal_alpha_mu(d, eta, beta, ksi, tau, L, Gamma, theta_val, g, c, mu_eff, y):
    denom = (tau * L + Gamma)
    if denom <= 1e-12:
        return 0.0
    alpha_min = 2.0 * (1.0 - eta) * beta * ksi * tau / denom
    a = max(alpha_min, 0.0)
    while (
        phi_mu(1.1 * a, eta, beta, tau, g, d, c, mu_eff, y, L, Gamma) < 0.0
        and (1.1 * a < alpha_min + theta_val * beta)
    ):
        a *= 1.1
    return float(a)

# ============================================================
# BLOCK-WISE ANCHOR SCALING
# ============================================================
def _block_slices():
    s = {}
    i = 0
    s["pde_phi"]  = slice(i, i + K_CON); i += K_CON
    s["pde_Ts"]   = slice(i, i + K_CON); i += K_CON
    s["pde_cont"] = slice(i, i + K_CON); i += K_CON
    s["pde_ru"]   = slice(i, i + K_CON); i += K_CON
    s["pde_rv"]   = slice(i, i + K_CON); i += K_CON
    s["pde_rTf"]  = slice(i, i + K_CON); i += K_CON

    s["bc_phi"]   = slice(i, i + N_BC_PHI); i += N_BC_PHI
    s["bc_solid"] = slice(i, i + N_BC_SOL); i += N_BC_SOL
    s["bc_fluid"] = slice(i, i + N_BC_FLD); i += N_BC_FLD

    s["iface"]    = slice(i, i + N_IF); i += N_IF
    s["ic"]       = slice(i, i + N_IC); i += N_IC
    assert i == M_CON, (i, M_CON)
    return s

BLOCKS = _block_slices()

@jax.jit
def row_rms(J_rows):
    return jnp.sqrt(jnp.mean(J_rows * J_rows, axis=1) + EPS)

@jax.jit
def make_block_row_scales(J, target_pde, target_bc, target_if, target_ic):
    s = jnp.ones((M_CON,), dtype=DTYPE)

    def set_block(name, target):
        sl = BLOCKS[name]
        rr = row_rms(J[sl, :])
        sb = target / jnp.maximum(target, rr)
        sb = jnp.clip(sb, S_ROW_MIN, S_ROW_MAX)
        return rr, sb

    rr_phi,  sb_phi  = set_block("pde_phi",  target_pde)
    rr_Ts,   sb_Ts   = set_block("pde_Ts",   target_pde)
    rr_cont, sb_cont = set_block("pde_cont", target_pde)
    rr_ru,   sb_ru   = set_block("pde_ru",   target_pde)
    rr_rv,   sb_rv   = set_block("pde_rv",   target_pde)
    rr_rTf,  sb_rTf  = set_block("pde_rTf",  target_pde)

    s = s.at[BLOCKS["pde_phi"]].set(sb_phi)
    s = s.at[BLOCKS["pde_Ts"]].set(sb_Ts)
    s = s.at[BLOCKS["pde_cont"]].set(sb_cont)
    s = s.at[BLOCKS["pde_ru"]].set(sb_ru)
    s = s.at[BLOCKS["pde_rv"]].set(sb_rv)
    s = s.at[BLOCKS["pde_rTf"]].set(sb_rTf)

    rr_bcphi, sb_bcphi = set_block("bc_phi", target_bc)
    rr_bcsol, sb_bcsol = set_block("bc_solid", target_bc)
    rr_bcfld, sb_bcfld = set_block("bc_fluid", target_bc)

    s = s.at[BLOCKS["bc_phi"]].set(sb_bcphi)
    s = s.at[BLOCKS["bc_solid"]].set(sb_bcsol)
    s = s.at[BLOCKS["bc_fluid"]].set(sb_bcfld)

    rr_if, sb_if = set_block("iface", target_if)
    rr_ic, sb_ic = set_block("ic", target_ic)

    s = s.at[BLOCKS["iface"]].set(sb_if)
    s = s.at[BLOCKS["ic"]].set(sb_ic)

    diag = jnp.array(
        [
            jnp.min(rr_phi), jnp.median(rr_phi), jnp.max(rr_phi),
            jnp.min(rr_rv),  jnp.median(rr_rv),  jnp.max(rr_rv),
            jnp.min(rr_bcfld), jnp.median(rr_bcfld), jnp.max(rr_bcfld),
            jnp.min(rr_if), jnp.median(rr_if), jnp.max(rr_if),
        ],
        dtype=DTYPE,
    )
    return s, diag

@jax.jit
def apply_row_scale(c, J, s_row):
    return s_row * c, s_row[:, None] * J

def _rms(x):
    return float(jnp.sqrt(jnp.mean(x * x) + 1e-30))

def _maxabs(x):
    return float(jnp.max(jnp.abs(x)))

def print_block_stats(prefix, c):
    def pr(name):
        sl = BLOCKS[name]
        v = c[sl]
        print(f"  {name:10s}: rms={_rms(v):.3e}  max={_maxabs(v):.3e}")

    print(prefix)
    pr("pde_phi")
    pr("pde_Ts")
    pr("pde_cont")
    pr("pde_ru")
    pr("pde_rv")
    pr("pde_rTf")
    pr("bc_phi")
    pr("bc_solid")
    pr("bc_fluid")
    pr("iface")
    pr("ic")

# ============================================================
# Training loop
# ============================================================
def train_sqp(seed=0,
              hidden_dim=35, num_hidden=3,
              max_iters=5000, print_every=10,
              beta_shift=200.0, beta_power=0.6,
              alpha_cap=1e1,
              L_lip=40.0, Gamma_lip=40.0,
              jitter_every_iter=True,
              sx=1e-5, sy=1e-5, st=1e-5,
              sx_ic=1e-5, sy_ic=1e-5,
              sy_if=1e-5, st_if=1e-5,
              w_pde_obj=10.0):

    key = random.PRNGKey(seed)

    layer_sizes = [3] + [hidden_dim] * num_hidden + [6]
    key, k0 = random.split(key, 2)
    params0 = init_mlp_params(k0, layer_sizes)
    theta, shapes = flatten_params(params0)

    n = theta.shape[0]
    H = jnp.eye(n, dtype=DTYPE)

    # objective points: force coverage in both regions
    key, k_obj_s, k_obj_f, k_con = random.split(key, 4)
    X_obj_s, _ = sample_stratified_3d(k_obj_s, x_min, Ls, y_min, y_max, t_min, t_max, NX_OBJ, NY_OBJ, NT_OBJ)
    X_obj_f, _ = sample_stratified_3d(k_obj_f, Ls, x_max, y_min, y_max, t_min, t_max, NX_OBJ, NY_OBJ, NT_OBJ)
    X_obj = jnp.concatenate([X_obj_s, X_obj_f], axis=0)

    X_con, ids_con = sample_stratified_3d(k_con, x_min, x_max, y_min, y_max, t_min, t_max, NX_CON, NY_CON, NT_CON)

    # boundary/interface/ic sets
    key, k1, k2, k3, k4, k5, k6, k7, k8, k9, k10, k11, k_ic = random.split(key, 13)

    X_phi_left,  ids_phi_left  = sample_bc_time_binned(k1, K_BC, x_fixed=x_min, y_lo=y_min, y_hi=y_max)
    X_phi_right, ids_phi_right = sample_bc_time_binned(k2, K_BC, x_fixed=x_max, y_lo=y_min, y_hi=y_max)
    X_phi_y0,    ids_phi_y0    = sample_bc_time_binned(k3, K_BC, y_fixed=y_min, x_lo=x_min, x_hi=x_max)
    X_phi_y1,    ids_phi_y1    = sample_bc_time_binned(k4, K_BC, y_fixed=y_max, x_lo=x_min, x_hi=x_max)

    X_Ts_x0, ids_Ts_x0 = sample_bc_time_binned(k5, K_BC, x_fixed=x_min, y_lo=y_min, y_hi=y_max)
    X_Ts_y0, ids_Ts_y0 = sample_bc_time_binned(k10, K_BC, y_fixed=y_min, x_lo=x_min, x_hi=Ls)
    X_Ts_y1, ids_Ts_y1 = sample_bc_time_binned(k11, K_BC, y_fixed=y_max, x_lo=x_min, x_hi=Ls)

    xL = float(Ls + X_MARGIN)
    xR = float(x_max - X_MARGIN)
    X_f_inlet,  ids_f_inlet  = sample_bc_time_binned(k6, K_BC, y_fixed=y_min, x_lo=xL, x_hi=xR)
    X_f_outlet, ids_f_outlet = sample_bc_time_binned(k7, K_BC, y_fixed=y_max, x_lo=xL, x_hi=xR)
    X_f_right,  ids_f_right  = sample_bc_time_binned(k8, K_BC, x_fixed=x_max, y_lo=y_min, y_hi=y_max)

    X_if, ids_if = sample_interface_yt(k9)
    X_ic, ids_ic = sample_ic_xy(k_ic)

    # line-search params (unchanged)
    eta, sigma = 0.25, 0.1
    eps_tau, eps_ksi = 1e-2, 1e-2
    theta_val = 10.0
    tau_k, ksi_k = 1.0, 1.0

    print("M_CON =", int(M_CON), "n_params =", int(n))
    print("PHI_SCALE =", float(PHI_SCALE), "FLUX_SCALE =", float(FLUX_SCALE))
    print("PHI_REF =", float(PHI_REF))
    print("T_SCALE =", float(T_SCALE), "U_SCALE =", float(U_SCALE), "P_SCALE =", float(P_SCALE))
    print("CONT_SCALE =", float(CONT_SCALE))
    print("Targets: PDE/BC/IF/IC =", float(S_TARGET_PDE), float(S_TARGET_BC), float(S_TARGET_IF), float(S_TARGET_IC))

    s_row_anchor = None

    t0_clock = time.time()
    for it in range(1, max_iters + 1):
        k_beta = (it // 10) * 10
        beta_k = float(min(1.0, (beta_shift / (beta_shift + k_beta)) ** beta_power))

        if jitter_every_iter:
            key, kj_obj, kj_con, kjL, kjR, kjB, kjT, kjSx, kjSy0, kjSy1, kjIn, kjOut, kjSym, kjIf, kjIc = random.split(key, 15)

            X_obj_use = jitter_xyt(kj_obj, X_obj, sx, sy, st)
            X_con_use = jitter_xyt(kj_con, X_con, sx, sy, st)

            X_phi_left_use  = project_x_fixed(jitter_xyt(kjL, X_phi_left,  0.0, sy, st), x_min)
            X_phi_right_use = project_x_fixed(jitter_xyt(kjR, X_phi_right, 0.0, sy, st), x_max)
            X_phi_y0_use    = project_y_fixed(jitter_xyt(kjB, X_phi_y0, sx, 0.0, st), y_min)
            X_phi_y1_use    = project_y_fixed(jitter_xyt(kjT, X_phi_y1, sx, 0.0, st), y_max)

            X_Ts_x0_use = project_x_fixed(jitter_xyt(kjSx, X_Ts_x0, 0.0, sy, st), x_min)
            X_Ts_y0_use = project_y_fixed(jitter_xyt(kjSy0, X_Ts_y0, sx, 0.0, st), y_min, x_lo=float(x_min), x_hi=float(Ls))
            X_Ts_y1_use = project_y_fixed(jitter_xyt(kjSy1, X_Ts_y1, sx, 0.0, st), y_max, x_lo=float(x_min), x_hi=float(Ls))

            X_f_inlet_use  = project_y_fixed(jitter_xyt(kjIn,  X_f_inlet,  sx, 0.0, st), y_min, x_lo=xL, x_hi=xR)
            X_f_outlet_use = project_y_fixed(jitter_xyt(kjOut, X_f_outlet, sx, 0.0, st), y_max, x_lo=xL, x_hi=xR)
            X_f_right_use  = project_x_fixed(jitter_xyt(kjSym, X_f_right,  0.0, sy, st), x_max)

            X_if_use = project_interface(jitter_xyt(kjIf, X_if, 0.0, sy_if, st_if))
            X_ic_use = project_t0(jitter_xyt(kjIc, X_ic, sx_ic, sy_ic, 0.0))
        else:
            X_obj_use = X_obj
            X_con_use = X_con
            X_phi_left_use, X_phi_right_use, X_phi_y0_use, X_phi_y1_use = X_phi_left, X_phi_right, X_phi_y0, X_phi_y1
            X_Ts_x0_use, X_Ts_y0_use, X_Ts_y1_use = X_Ts_x0, X_Ts_y0, X_Ts_y1
            X_f_inlet_use, X_f_outlet_use, X_f_right_use = X_f_inlet, X_f_outlet, X_f_right
            X_if_use, X_ic_use = X_if, X_ic

        obj_val, g = F_and_g_pde_obj(theta, shapes, X_obj_use)
        g = DTYPE(w_pde_obj) * g

        args = (
            X_con_use, ids_con,
            X_phi_left_use, ids_phi_left,
            X_phi_right_use, ids_phi_right,
            X_phi_y0_use, ids_phi_y0,
            X_phi_y1_use, ids_phi_y1,
            X_Ts_x0_use, ids_Ts_x0,
            X_Ts_y0_use, ids_Ts_y0,
            X_Ts_y1_use, ids_Ts_y1,
            X_f_inlet_use, ids_f_inlet,
            X_f_outlet_use, ids_f_outlet,
            X_f_right_use, ids_f_right,
            X_if_use, ids_if,
            X_ic_use, ids_ic,
        )

        c, J = C_and_J(theta, shapes, *args)

        if s_row_anchor is None:
            s_row_anchor, diag = make_block_row_scales(J, S_TARGET_PDE, S_TARGET_BC, S_TARGET_IF, S_TARGET_IC)

            print("[anchor] applied block-wise row scaling.")
            print(
                "  s_row min/med/max:",
                float(jnp.min(s_row_anchor)),
                float(jnp.median(s_row_anchor)),
                float(jnp.max(s_row_anchor)),
            )
            print("  s_row fraction<1:", float(jnp.mean((s_row_anchor < 0.999).astype(DTYPE))))
            print("  diag(row_rms) [phi min/med/max, rv min/med/max, bc_fluid min/med/max, iface min/med/max]:")
            print(" ", [float(x) for x in diag])

        c_s, J_s = apply_row_scale(c, J, s_row_anchor)

        d, y_s = kkt_solve_once(H, J_s, g, c_s, MU_DAMP_FIXED)

        # map multipliers back
        y = s_row_anchor * y_s

        dn = float(jnp.linalg.norm(d, jnp.inf))
        if dn > 1e-12:
            tau_k = cal_tau_mu(H, d, sigma, tau_k, eps_tau, g, c_s, MU_DAMP_FIXED, y_s)
            ksi_k = cal_ksi_mu(d, tau_k, ksi_k, eps_ksi, g, c_s, MU_DAMP_FIXED, y_s)
            alpha = cal_alpha_mu(d, eta, beta_k, ksi_k, tau_k, L_lip, Gamma_lip, theta_val, g, c_s, MU_DAMP_FIXED, y_s)
        else:
            alpha = 0.0

        alpha = float(min(alpha, alpha_cap))
        theta = theta + DTYPE(alpha) * d

        if it % int(print_every) == 0:
            c_now = C_only(theta, shapes, *args)

            feas_raw = float(jnp.mean(c_now ** 2))
            feas_s   = float(jnp.mean((s_row_anchor * c_now) ** 2))
            station  = float(jnp.linalg.norm(g + J.T @ y, jnp.inf))

            print(
                f"[it={it}] obj={float(w_pde_obj * obj_val):.3e} feas_raw={feas_raw:.3e} "
                f"feas_s={feas_s:.3e} alpha={alpha:.2e} beta={beta_k:.2e} "
                f"station={station:.2e} tau={tau_k:.3e} dn={dn:.2e}"
            )

            print_block_stats("blocks (unscaled c):", c_now)

            params_now = unflatten_params(theta, shapes)
            r_phi, r_Ts, r_cont, r_u, r_v, r_Tf, ms, mf = residuals_all(params_now, X_obj_use)
            mf_obj = (X_obj_use[:, 0] > Ls).astype(DTYPE)
            ms_obj = DTYPE(1.0) - mf_obj
            print(
                "PDE RMS on obj:",
                "phi", _rms(r_phi),
                "Ts(solid)", _rms(ms_obj * r_Ts),
                "cont(fluid)", _rms(mf_obj * r_cont),
                "ru(fluid)", _rms(mf_obj * r_u),
                "rv(fluid)", _rms(mf_obj * r_v),
                "rTf(fluid)", _rms(mf_obj * r_Tf),
            )

    print(f"[done] elapsed={time.time() - t0_clock:.2f}s")
    return theta

# ============================================================
# Run
# ============================================================
def main():
    theta = train_sqp(
        seed=0,
        hidden_dim=35,
        num_hidden=3,
        max_iters=20000,
        print_every=10,
        beta_shift=400.0,
        beta_power=0.6,
        alpha_cap=1e1,
        L_lip=10.0,
        Gamma_lip=10.0,
        jitter_every_iter=True,
        sx=2e-7,#try 2e-7
        sy=1e-6,#2e-6
        st=1e-5,#5e-5
        sx_ic=2e-7,
        sy_ic=1e-5,
        sy_if=1e-6,
        st_if=1e-5,
        w_pde_obj=10.0,
    )
    np.save("theta_sqp_pinn_exp2_block_anchor_scaled.npy", np.array(theta))
    print("Saved theta -> theta_sqp_pinn_exp2_block_anchor_scaled.npy")

if __name__ == "__main__":
    main()

Loaded phi.txt: (65,) (16,) (65, 16) range= 0.5 6.2
M_CON = 2592 n_params = 2876
PHI_SCALE = 1081.5332578341051 FLUX_SCALE = 194095.13007348892
PHI_REF = 7.0
T_SCALE = 100.0 U_SCALE = 0.400000000001 P_SCALE = 1775.3600000000013
CONT_SCALE = 35.08771929217759
Targets: PDE/BC/IF/IC = 1.0 1.0 1.0 1.0
[anchor] applied block-wise row scaling.
  s_row min/med/max: 0.2140773895318467 1.0 1.0
  s_row fraction<1: 0.011574074074074073
  diag(row_rms) [phi min/med/max, rv min/med/max, bc_fluid min/med/max, iface min/med/max]:
  [0.00962390318144111, 0.019232701120766076, 0.12412863380867056, 1e-06, 0.010327352417367052, 0.11334091044104165, 7.640696595966051e-05, 0.019119245147106813, 0.14080741058792381, 0.00017590674776668205, 0.023914693947328414, 0.12928520679172287]
[it=10] obj=7.174e+01 feas_raw=1.041e+00 feas_s=2.140e-01 alpha=4.90e-02 beta=9.85e-01 station=5.47e+00 tau=1.893e-01 dn=5.47e+00
blocks (unscaled c):
  pde_phi   : rms=1.563e-01  max=3.286e-01
  pde_Ts    : rms=1.852e-01  max=4.